In [1]:
import pandas as pd
from lca import *
from utils import build_full_dict, question_list, analisis_estadistico_clusters, analisis_interacciones_variables

In [2]:
df_sintomas_disc = pd.read_excel(r"data\Sintomas_disc_Infante_Juvenil_n=1558.xlsx")
df_etiquetas = pd.read_excel(r"data\preguntas_disc.xlsx")
df_base = pd.read_excel(r"data\base_innominada.xlsx")

In [ ]:
full_dict = build_full_dict(df_sintomas_disc)
q_list = question_list()

In [ ]:
df_sintomas_con_edad = pd.merge(
    df_sintomas_disc,
    df_base[['inum', 'ed1']],
    left_on='id',
    right_on='inum',
    how='left'
)

df_sintomas_con_edad = df_sintomas_con_edad.drop(columns=['inum'])

df_child = df_sintomas_con_edad[df_sintomas_con_edad['ed1'] <= 11].copy()
df_teen = df_sintomas_con_edad[df_sintomas_con_edad['ed1'] > 11].copy()

print('# Sujetos y columnas DF Child')
display(df_teen.shape)

df=df_teen.copy()
output_df, prob_df, metric_df = perform_true_lca(
    df=df,
    question_list = q_list,
    id_column='id',
    n_components_range=range(2, 7)
)

In [ ]:
from utils import build_full_dict, question_list, analisis_estadistico_clusters, analisis_interacciones_variables

resultados = analisis_estadistico_clusters(
    final_df=output_df,
    id_column='id',
    cluster_prefix='lca_k',
    df_etiquetas=df_etiquetas,
    col_codigo='Código',
    col_etiqueta='Etiqueta',
    output_pdf='reporte_teen.pdf',
    metrics_df=metric_df
)

excluir = ['id'] + [col for col in output_df.columns if col.startswith('lca_k')]
cramersv_mat, pvalue_mat, chi2_mat = analisis_interacciones_variables(
    df=output_df,
    exclude_cols=excluir
)

from utils import guardar_df

# Guardar DataFrames principales
guardar_df(output_df, 'LCA_teen_n', 'LCA_n')
guardar_df(prob_df, 'LCA_prob_teen_n', 'LCA_prob_n')
guardar_df(metric_df, 'LCA_metric_teen_n', 'LCA_metric_n')

# Guardar matrices de interacción (son DataFrames cuadrados)
guardar_df(cramersv_mat, 'cramersv_teen', 'cramers_matrix')
guardar_df(pvalue_mat, 'pvalue_teen', 'pvalue_matrix')
guardar_df(chi2_mat, 'chi2_teen', 'resultados_matrix')

# Guardar las tablas completas de Cramér's V para cada k
for i in range(2, 7):
    nombre_k = f'lca_k{i}'
    if nombre_k in resultados['cramers_completos']:
        df_cramers = resultados['cramers_completos'][nombre_k]
        guardar_df(df_cramers, f'Cramers_teen_{nombre_k}', 'Cramers_lca')
    else:
        print(f"Advertencia: {nombre_k} no encontrado en resultados['cramers_completos']")

In [ ]:
import winsound
duration = 1000  # milisegundos
freq = 700       # Hz
winsound.Beep(freq, duration)
winsound.Beep(freq, duration)
winsound.Beep(freq, duration)